# Tutorial: Interactive HUC map selection

Use an interactive map to select a HUC6 or HUC8 region and obtain the STREAM monitoring sites inside it. The map loads site points first and requests only the boundary that you click. It does **not** download hundreds of HUC polygons before displaying the map.

## 1. Connect to STREAM

In [ ]:
from stream_utils.client import StreamClient
from stream_utils.huc_map_selector import build_huc_selection_map

stream = StreamClient()

## 2. Choose which sites to display

Start with the small site metadata table. You can apply any `find_sites()` filters before building the map. Here, the map shows sites that have water-temperature observations.

In [ ]:
sites = stream.find_sites(variables="WTemp_C")
print(f"Displaying {len(sites):,} STREAM sites")
sites.head()

## 3. Display the selection map

Choose `HUC6` for larger basins or `HUC8` for smaller subbasins. Run the cell and then click anywhere on the map. The selected boundary will be downloaded and highlighted after the click.

In [ ]:
huc_map, selection = build_huc_selection_map(
    sites,
    huc_level="HUC8",
)
huc_map

## 4. Inspect the selected sites

After clicking a HUC on the map, run this cell. The `selection` dictionary is updated automatically and can be reused in later analyses.

In [ ]:
if selection["sites"] is None:
    print("Click a location on the map first.")
else:
    selected_sites = selection["sites"]
    print(f"Selected HUC: {selection['huc_id']}")
    print(f"Selected sites: {len(selected_sites):,}")
    display(selected_sites.head())

## 5. Read data for the selection

The selected site identifiers can be passed directly to the regular STREAM data-access functions. This example requests one month of water temperature.

In [ ]:
if selection["sites"] is not None and not selection["sites"].empty:
    site_ids = selection["sites"]["STREAM_ID"].tolist()
    water_quality = stream.read_water_quality(
        site_ids=site_ids,
        variables="WTemp_C",
        start="2020-06-01",
        end="2020-06-30",
    )
    print(f"Retrieved {len(water_quality):,} observations")
    display(water_quality.head())